# Sprint 3 - Augmentation & Better Training

**Goal (see AGENT.md / final_brief_and_plan.md):** make the PlantVillage-trained model more robust.

- Add the **Albumentations** pipeline (flips, rotation, brightness/contrast, blur, perspective,
  JPEG-compression artifacts) to the training split only
- Retrain the same frozen-head setup WITH augmentation
- Compare against the Sprint 1 baseline (0.9613 acc / 0.9501 F1) and log the run
- **Done when:** `augmentation_only` row in the ablation CSV and we can see whether aug helps

This is ablation **variant 2 of 4**. Variant 3 (PlantDoc fine-tuning) and 4 (both) come in Sprint 4.
Validation/test are NEVER augmented — evaluation stays deterministic, same as baseline.

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

Requires the Sprint 0 archives (`plantvillage_raw.zip` + manifest) to be on Drive and verified.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")    # durable archives (from Sprint 0)
LOCAL_RAW_DIR = Path("/content/folium_raw")             # per-session raw
LOCAL_DATA_DIR = Path("/content/folium_data")            # per-session organized splits
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 3 - Hydrate raw from Drive, then organize splits locally

Same as Sprint 0: unzip the two archives from Drive into local raw, then build
`train/val/test` folders with `scripts/organize_datasets.py`. Deterministic and
idempotent, so re-running is harmless.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed"
print("splits ready at", LOCAL_DATA_DIR)

## Step 4 - Train WITH augmentation (frozen head, 10 epochs)

Same Stage 1 setup as Sprint 1 (`python -m ml.train --augment`) but with the augmentation pipeline
and more epochs (augmentation fights memorization, so you can train longer safely). Artifacts are
tagged `stage1_aug` so they never overwrite the baseline checkpoints. Run 4-5 epochs first if the
GPU session is tight - the comparison still works.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--augment",
    "--tag", "stage1_aug",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.train failed"
print("training finished; best checkpoint:", CHECKPOINT_DIR / "best_plantvillage_stage1_aug.pt")

## Step 5 - Evaluate the augmented model on the test set

Appends the `augmentation_only` row to the ablation CSV and writes the confusion matrix (overwrites
the baseline PNG - the per-run numbers live in the CSV).

In [ ]:
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_stage1_aug.pt"),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--split", "test",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
    "--confusion-path", str(RESULTS_DIR / "confusion_matrix.png"),
    "--variant", "augmentation_only",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.evaluate failed"

import pandas as pd
ablation = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
print(ablation[["variant", "dataset", "accuracy", "precision", "recall", "f1"]].to_string(index=False))

## Step 6 - Baseline vs augmentation (the Sprint 3 verdict)

Both rows are now in the CSV: `baseline_pv_only_no_aug` (Sprint 1) vs `augmentation_only` (this run).
Watch accuracy, and especially recall/F1 - augmentation usually shows up on the messier classes first.

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
key = ["variant", "accuracy", "precision", "recall", "f1"]
compare = df[df["variant"].isin(["baseline_pv_only_no_aug", "augmentation_only"])][key]
print(compare.to_string(index=False))

if len(compare) == 2:
    aug = compare[compare["variant"] == "augmentation_only"].iloc[0]
    base = compare[compare["variant"] == "baseline_pv_only_no_aug"].iloc[0]
    print(f"\nverdict: accuracy {'+%.4f' % (aug['accuracy'] - base['accuracy'])} | f1 {'+%.4f' % (aug['f1'] - base['f1'])} (augmented - baseline)")
    print("Sprint 3 DONE" if aug["f1"] >= base["f1"] else "Sprint 3 NOT met yet - aug did not beat baseline")

## Step 7 - Predict with the augmented model

Same done-when check as Sprint 1, now with the `stage1_aug` checkpoint.

In [ ]:
from pathlib import Path

images = sorted((LOCAL_DATA_DIR / "plantvillage" / "test").glob("*/*.jpg"))
for image in images[:3]:
    result = subprocess.run([
        sys.executable, "-m", "ml.predict",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_stage1_aug.pt"),
        "--image", str(image),
    ], cwd=str(REPO_DIR))
    assert result.returncode == 0, "ml.predict failed"
    print()

## Where things live

**On Google Drive (durable):**
```
folium/checkpoints/plantvillage_stage1_aug_epoch*.pt   one per epoch (augmented run)
folium/checkpoints/best_plantvillage_stage1_aug.pt     best on validation (augmented)
folium/checkpoints/best_plantvillage_stage1.pt         Sprint 1 baseline (untouched)
folium/results/ablation_results.csv                    now holds 2 of the 4 ablation rows
folium/results/confusion_matrix.png                    last run's matrix (augmentation_only)
```

**CLI:** `python -m ml.train ... [--augment --tag <name>]` and
`python -m ml.evaluate ... --variant <name>` run on Colab or locally.

Sprint 4 adds variants 3 and 4: PlantDoc fine-tuning (Stage 2) and augmentation + PlantDoc together.